# 04 — Gaussian Case, and $K$-step Sinkhorn analysis

- In part A, we analyze the case where $p$ and $q$ are Gaussians, the different sources of error and effect of bandwidth on the convergence rate.
- In part B, we study `sinkhorn_drift(sigma, K)`, local rate $\Gamma_K$ and tradeoff local convregence speed vs computationnal cost.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from msflow import *
from msflow import gaussian as gs
import msflow

pd.set_option("display.width", 180)

---
## A1 · Particles versus the closed moment map (sanity check)

In [ ]:
data = load_dataset("gaussian_aniso", n=2000, seed=2026)
SIGMA, TAU, N_ITER = 1.0, 0.05, 400
m_star, S_star = data.p.mean(), data.p.cov()
m0, S0 = data.q0.mean(), data.q0.cov()
print("target  mean", np.round(m_star, 3), " eigenvalues", np.round(np.linalg.eigvalsh(S_star), 3))
print("source  mean", np.round(m0, 3), " eigenvalues", np.round(np.linalg.eigvalsh(S0), 3))

field = mean_shift(get_kernel("Gaussian", SIGMA))
part = run(data.q0, data.p, field, n_iter=N_ITER, tau=TAU, clip=None,
           probes={"W2": lambda k, p, q, V: W2(q, p)}, probe_every=10,
           store_every=5, label="particles")

theory_map = gs.moment_flow(m0, S0, m_star, S_star, SIGMA, TAU, N_ITER, scheme="map")
theory_ode = gs.moment_flow(m0, S0, m_star, S_star, SIGMA, TAU, N_ITER, scheme="ode")

# the cloud read through its own Gaussian moments, to compare like with like
ks_p, w2_moments = replay(part, lambda q: gs.W2_gaussian(q.mean(), q.cov(), m_star, S_star))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(9.6, 3.2))
ax[0].semilogy(part.probe_k, part.probes["W2"], lw=2.4, alpha=.65,
               color="#d62728", label="particles, exact $W_2$")
ax[0].semilogy(ks_p, w2_moments, lw=1.2, color="#8c564b", ls="-.",
               label="particles, Gaussian moments")
ax[0].semilogy(theory_map["W2"], lw=1.6, ls="--", color="#1f77b4", label="moment map (exact)")
ax[0].semilogy(theory_ode["W2"], lw=1.6, ls=":", color="#2ca02c", label="moment ODE (Euler)")
ax[0].set(xlabel="$k$", ylabel=r"$W_2(q^k,p)$", title="particles vs closed form")
ax[0].legend(fontsize=7.5)

ax[1].semilogy(theory_map["Phi"], lw=1.8, color="#1f77b4", label="map")
ax[1].semilogy(theory_ode["Phi"], lw=1.8, ls=":", color="#2ca02c", label="ODE")
ax[1].set(xlabel="$k$", ylabel=r"$\Phi(\Sigma_k)$",
          title=r"Lyapunov $\Phi=2\,\mathrm{KL}$ (LaSalle)")
ax[1].legend(fontsize=8)

ev_map = np.array([np.linalg.eigvalsh(S) for S in theory_map["Sigma"]])
ev_star = np.linalg.eigvalsh(S_star)
for j in range(2):
    ax[2].plot(ev_map[:, j], lw=1.8, color=f"C{j}", label=rf"$\lambda_{j+1}(\Sigma_k)$")
    ax[2].axhline(ev_star[j], ls="--", lw=1, color=f"C{j}")
ax[2].set(xlabel="$k$", ylabel="eigenvalue", title=r"$\Sigma_k\to\Sigma^\star$ (dashed)")
ax[2].legend(fontsize=8)
for a in ax: a.grid(alpha=.3)
fig.tight_layout(); save(fig, "04_gaussian_validation")
print("Phi is non-increasing:", bool(np.all(np.diff(theory_map["Phi"]) <= 1e-12)))

### Two errors, with limited evidence for their slopes

Increasing particle count should reduce the empirical Gaussian-field error;
decreasing $\tau$ at fixed physical time should reduce map-versus-ODE error
at first order.  The log-log slopes below use a small grid and few seeds, so
treat the reference lines as consistency checks rather than precise
non-asymptotic rate estimates.  This calibration does not validate
non-Gaussian kernels, curl/gauge diagnostics or global certificates.

In [ ]:
# (a) Monte-Carlo: how fast does the particle flow approach the moment map?
NS, SEEDS, K_CMP = [500, 1000, 1500, 2000, 3000], [0, 1, 2, 3], 100

def mc_gap(n, seed):
    d = load_dataset("gaussian_aniso", n=n, seed=seed)
    r = run(d.q0, d.p, mean_shift(get_kernel("Gaussian", SIGMA)),
            n_iter=K_CMP, tau=TAU, clip=None, probes={}, store_every=K_CMP)
    th = gs.moment_flow(d.q0.mean(), d.q0.cov(), d.p.mean(), d.p.cov(),
                        SIGMA, TAU, K_CMP, scheme="map")
    S_th = th["Sigma"][-1]
    return np.linalg.norm(r.q_final.cov() - S_th, "fro") / np.linalg.norm(S_th, "fro")

gaps = [float(np.mean([mc_gap(n, s) for s in SEEDS])) for n in NS]

# (b) discretization: gap between the exact map and the Euler ODE as tau shrinks
TAUS = np.array([0.005, 0.01, 0.02, 0.05, 0.1])
tau_gaps = []
for t in TAUS:
    n_it = int(round(2.0 / t))                     
    a = gs.moment_flow(m0, S0, m_star, S_star, SIGMA, t, n_it, scheme="map")
    b = gs.moment_flow(m0, S0, m_star, S_star, SIGMA, t, n_it, scheme="ode")
    tau_gaps.append(abs(a["W2"][-1] - b["W2"][-1]) / a["W2"][-1])

slope_n = np.polyfit(np.log(NS), np.log(gaps), 1)[0]
slope_t = np.polyfit(np.log(TAUS), np.log(tau_gaps), 1)[0]

fig, ax = plt.subplots(1, 2, figsize=(9.0, 3.3))
ax[0].loglog(NS, gaps, "o-", lw=1.8, color="#d62728", label=f"measured (slope {slope_n:.2f})")
ax[0].loglog(NS, gaps[0] * (np.array(NS) / NS[0]) ** -0.5, ls="--", color="0.5",
             label=r"$n^{-1/2}$ guide")
ax[0].set(xlabel="particles $n$", ylabel=r"$\|\Sigma_{\rm part}-\Sigma_{\rm theory}\|_F/\|\Sigma_{\rm theory}\|_F$",
          title=f"(a) Monte-Carlo error at $k={K_CMP}$")
ax[0].legend(fontsize=8)
ax[1].loglog(TAUS, tau_gaps, "o-", lw=1.8, color="#2ca02c",
             label=f"measured (slope {slope_t:.2f})")
ax[1].loglog(TAUS, tau_gaps[-1] * (TAUS / TAUS[-1]), ls="--", color="0.5",
             label=r"$\tau$ guide")
ax[1].set(xlabel=r"step size $\tau$", ylabel="relative gap, map vs ODE",
          title="(b) discretization error")
ax[1].legend(fontsize=8)
for a in ax: a.grid(alpha=.3, which="both")
fig.tight_layout(); save(fig, "04_gaussian_error_rates")

conv = pd.DataFrame({"n": NS, "MC gap": gaps}).set_index("n")
tconv = pd.DataFrame({"tau": TAUS, "map vs ODE gap": tau_gaps}).set_index("tau")
save_table(conv, "04_mc_convergence",
           caption=rf"Monte-Carlo convergence to the moment map (fitted slope {slope_n:.2f}, "
                   r"expected $-1/2$).")
save_table(tconv, "04_tau_convergence",
           caption=rf"Discretization gap, exact moment map vs Euler moment ODE "
                   rf"(fitted slope {slope_t:.2f}, expected $1$).")
print(f"Monte-Carlo slope {slope_n:.3f} (expected -0.5);  "
      f"discretization slope {slope_t:.3f} (expected 1)")
display(conv.round(5), tconv.round(6))

## A2 · Gaussian bandwidth sweep

In [ ]:
sig_rows = []
for s in [0.25, 0.5, 1.0, 2.0, 4.0]:
    fl = gs.moment_flow(m0, S0, m_star, S_star, s, TAU, 1500, scheme="map")
    sig_rows.append({"sigma": s, "W2 at k=1500": fl["W2"][-1],
                     "Phi at k=1500": fl["Phi"][-1],
                     "fitted rate": fit_exponential_rate(np.arange(len(fl["W2"])), fl["W2"])})
sig_tab = pd.DataFrame(sig_rows).set_index("sigma")
save_table(sig_tab, "04_gaussian_sigma",
           caption=r"Gaussian-Gaussian flow: effect of the bandwidth on the exact "
                   r"moment dynamics.")
sig_tab.round(5)

---
## B1 · Local Gaussian rate $\Gamma_K$

In [ ]:
alphas = np.logspace(-1.5, 2, 300)
Ks = [0, 1, 2, 5, 10, 20, -1]

fig, ax = plt.subplots(1, 2, figsize=(9.0, 3.4))
for K in Ks:
    lab = r"$K=\infty$" if K < 0 else f"$K={K}$"
    ax[0].semilogx(alphas, gs.gamma_K(alphas, K), lw=1.8,
                   ls="--" if K < 0 else "-", label=lab)
ax[0].set(xlabel=r"$\alpha=\lambda/\sigma^2$", ylabel=r"$\Gamma_K$",
          title=r"local rate: a narrower kernel needs more corrections", ylim=(0, 1.05))
ax[0].legend(fontsize=8, ncol=2)

lam, sig_scan = 1.0, np.logspace(-1, 1, 60)
for K in [0, 2, 10, -1]:
    ana = gs.gamma_K(lam / sig_scan ** 2, K)
    num = [gs.gamma_K_numeric(lam, s, K) for s in sig_scan]
    lab = r"$K=\infty$" if K < 0 else f"$K={K}$"
    ax[1].semilogx(sig_scan ** 2, ana, lw=1.8, label=lab)
    ax[1].semilogx(sig_scan ** 2, num, ls="none", marker=".", ms=4, color="k")
ax[1].plot([], [], ls="none", marker=".", color="k", label=r"$-F_K'(\lambda)$ (finite diff.)")
ax[1].set(xlabel=r"$\sigma^2$  (at $\lambda=1$)", ylabel=r"$\Gamma_K$",
          title="closed-form recursion vs finite differences")
ax[1].legend(fontsize=8)
for a in ax: a.grid(alpha=.3, which="both")
fig.tight_layout(); save(fig, "04_gamma_K");

In [ ]:
table = gs.rate_table(Ks=(0, 1, 2, 5, 10, 20, 50))
save_table(
    table, "04_rate_table",
    caption=("Locally optimized Gaussian bandwidth/rate versus K, with "
             "Gamma_K divided by the idealized scaling count 2K+1."))
table.round(4)

If a fixed overhead $r$ is added to the proxy cost, correction $K+1$ beats
$K$ exactly when

$$\frac{\Gamma^\star_{K+1}}{r+2K+3}>
  \frac{\Gamma^\star_K}{r+2K+1}.$$

The next panel displays this model's break-even $r_K$.  It remains an
analytic scaling-update proxy, not measured compute or a universal optimum.

In [ ]:
Gs = [gs.optimal_alpha(K)[1] for K in range(0, 12)]
rK = [2 * Gs[K] / (Gs[K + 1] - Gs[K]) - (2 * K + 1)
      for K in range(len(Gs) - 1)]
overhead = pd.DataFrame(
    {"Gamma_K*": Gs[:-1], "Gamma_{K+1}*": Gs[1:],
     "break-even overhead r_K": rK},
    index=pd.Index(range(len(rK)), name="K"))

fig, ax = plt.subplots(1, 2, figsize=(9.0, 3.3))
for overhead_ratio in [0, 2, 10, 50, 200]:
    proxy_rates = [g / (overhead_ratio + 2 * K + 1) for K, g in enumerate(Gs)]
    ax[0].plot(range(len(Gs)), proxy_rates, marker="o", ms=4, lw=1.6,
               label=f"$r={overhead_ratio}$")
    best = int(np.argmax(proxy_rates))
    ax[0].scatter([best], [proxy_rates[best]], s=110, facecolor="none",
                  edgecolor="k", lw=1.4, zorder=5)
ax[0].set(xlabel="$K$", ylabel=r"$\Gamma_K^\star/(r+2K+1)$", yscale="log",
          title=r"local rate per scaling-count proxy")
ax[0].legend(fontsize=8, title="fixed overhead ratio")
ax[1].semilogy(overhead.index, overhead["break-even overhead r_K"],
               marker="o", lw=1.8)
ax[1].set(xlabel="$K$", ylabel=r"$r_K$",
          title=r"proxy overhead where correction $K+1$ pays")
for a in ax: a.grid(alpha=.3)
fig.tight_layout(); save(fig, "04_cost_tradeoff")

save_table(overhead, "04_cost_tradeoff",
           caption="Break-even overhead ratio in the idealized scaling-count model.")
overhead.round(3)

## B2 · Dissipation retained in the commuting Gaussian model

The monotonic $\alpha_K$ statement is specific to the centered commuting
Gaussian calculation.  A “realized minimum” in the table is a minimum over
the displayed finite grid of covariances, not an infimum over all $q$.

In [ ]:
lam = np.array([2.0, 0.5])
qs = np.logspace(-2, 1.2, 160)
fig, ax = plt.subplots(1, 2, figsize=(9.0, 3.4))

for K in [0, 1, 2, 5, 10]:
    a = [gs.dissipation_fraction(lam, np.array([v, v * 0.4]), 1.0, K) for v in qs]
    ax[0].semilogx(qs, a, lw=1.8, label=f"$K={K}$")
ax[0].axhline(1, color="k", ls="--", lw=1, label=r"$\alpha_\infty=1$")
ax[0].set(xlabel=r"$q_1$ (model covariance)", ylabel=r"$\alpha_K(p,q)$",
          title=r"fraction of the $S_\varepsilon$ dissipation retained", ylim=(0, 1.05))
ax[0].legend(fontsize=8, ncol=2)

kappa = np.logspace(-1.5, 2, 200)          # lambda_max / sigma^2
ax[1].loglog(kappa, [gs.alpha_0_lower_bound(k, 1.0) for k in kappa], lw=2,
             color="#d62728", label=r"$\underline{\alpha}_0(p)$ (uniform bound)")
worst = [min(gs.dissipation_fraction(np.array([k]), np.array([v]), 1.0, 0)
             for v in np.logspace(-3, 3, 90)) for k in kappa]
ax[1].loglog(kappa, worst, lw=1.6, ls="--", color="#1f77b4",
             label=r"realized $\min_q \alpha_0$")
ax[1].axvline(1.0, color="0.5", ls=":", lw=1)
ax[1].set(xlabel=r"$\kappa=\lambda_{\max}/\sigma^2$", ylabel=r"$\alpha_0$",
          title=r"population K=0: low-temperature bound")
ax[1].legend(fontsize=8)
for a in ax: a.grid(alpha=.3, which="both")
fig.tight_layout(); save(fig, "04_alpha_K")

alpha_tab = pd.DataFrame(
    {f"K={K}": [gs.dissipation_fraction(np.array([1.0]), np.array([v]), s, K)
                for s in [0.3, 1.0, 3.0] for v in [0.2, 5.0]]
     for K in [0, 1, 2, 5, 10]},
    index=pd.MultiIndex.from_product([[0.3, 1.0, 3.0], [0.2, 5.0]], names=["sigma", "q"]))
save_table(alpha_tab, "04_alpha_K",
           caption=r"Dissipation fraction $\alpha_K$ retained by $K$ corrections "
                   r"($\lambda=1$).")
alpha_tab.round(4)

---
## C · Extension to Non-Gaussian particles

In [ ]:
ring = load_dataset("ring", n=250, seed=2026)
SIGMA_R, TAU_R, N_R = 1.0, 0.04, 300
KS = [0, 1, 2, 5, 10, 20, -1]

sink_runs = Runs()
for K in KS:
    key = "Kinf" if K < 0 else f"K{K}"
    sink_runs[key] = run(
        ring.q0, ring.p, sinkhorn_drift(SIGMA_R, K=K),
        n_iter=N_R, tau=TAU_R, clip=None,
        probes=standard_probes(ring, get_kernel("Gaussian", SIGMA_R)),
        probe_every=25, store_every=50,
        label=(r"$K=\infty$" if K < 0 else f"$K={K}$"))

reference_runs = Runs({
    "deng_official": run(
        ring.q0,
        ring.p,
        deng_official(
            temperature=SIGMA_R,
            self_mask=True,
            self_mask_distance=1e6,
        ),
        n_iter=N_R,
        tau=TAU_R,
        clip=None,
        probes=standard_probes(ring),
        probe_every=25,
        store_every=50,
        label="Deng Algorithm 2",
    ),
    
    "sinkhorn_proxy": run(ring.q0, ring.p,
                          sinkhorn_proxy(get_kernel("Gaussian", SIGMA_R)),
                          n_iter=N_R, tau=TAU_R, clip=None,
                          probes=standard_probes(ring), probe_every=25,
                          store_every=50, label="Sinkhorn Proxy"),
})
comparison_runs = Runs({**sink_runs, **reference_runs})
fig = plot_runs(comparison_runs, keys=("W2", "H", "n_covered", "entropy"))
save(fig, "04_sinkhorn_particles_iter");

In [ ]:
def scaling_cost(r):
    K = r.meta.get("K")
    return 2 * K + 1 if K is not None and K >= 0 else np.nan

sink_tab = runs_table(
    comparison_runs, columns=("W2", "H", "n_covered", "entropy", "precision"),
    n_modes=len(ring.centers), extra=lambda r: {"scaling-count proxy": scaling_cost(r)})
save_table(
    sink_tab, "04_sinkhorn_particles",
    caption=("Population K-family plus external Deng/Proxy references on the ring "
             f"(sigma={SIGMA_R:g} only for Gaussian fields, tau={TAU_R:g}, "
             f"n_iter={N_R})."))
sink_tab.round(4)

### Fixed budget analysis

In [ ]:
BUDGET = 4200
BUDGET_KS = [K for K in KS if K >= 0]
budget_runs = Runs()
for K in BUDGET_KS:
    n_it = BUDGET // (2 * K + 1)
    budget_runs[f"B_K{K}"] = run(
        ring.q0, ring.p, sinkhorn_drift(SIGMA_R, K=K),
        n_iter=n_it, tau=TAU_R, clip=None,
        probes={"W2": lambda kk, p, q, V: W2(q, p),
                "cov": lambda kk, p, q, V: mode_metrics(q, ring.centers)["n_covered"]},
        probe_every=max(n_it // 25, 1), store_every=n_it,
        label=rf"$K={K}$ ({n_it} it.)")

fig, ax = plt.subplots(1, 2, figsize=(9.0, 3.4))
for key, r in budget_runs.items():
    K = r.meta["K"]
    ax[0].loglog(np.maximum(r.probe_k * (2 * K + 1), 1), r.probes["W2"],
                 color=color_of(key), lw=1.7, label=r.label)
ax[0].set(xlabel=r"scaling-count proxy $k(2K+1)$", ylabel=r"$W_2$",
          title=f"fixed proxy budget B={BUDGET}")
ax[0].legend(fontsize=8)
Ks_b = [r.meta["K"] for r in budget_runs.values()]
ax[1].semilogy(Ks_b, [r.final("W2") for r in budget_runs.values()],
               marker="o", lw=1.8, color="#1f77b4")
ax[1].set(xlabel="$K$", ylabel=r"final $W_2$",
          title="best sampled K under the proxy budget")
for a in ax: a.grid(alpha=.3, which="both")
fig.tight_layout(); save(fig, "04_sinkhorn_particles_cost")

budget_tab = pd.DataFrame(
    {"outer iterations": [r.n_iter for r in budget_runs.values()],
     "scaling proxy/step": [2 * r.meta["K"] + 1 for r in budget_runs.values()],
     "final W2": [r.final("W2") for r in budget_runs.values()],
     "modes reached": [int(r.probes["cov"].max()) for r in budget_runs.values()]},
    index=pd.Index(Ks_b, name="K"))
save_table(budget_tab, "04_sinkhorn_budget",
           caption=f"K-family under fixed idealized scaling-count budget {BUDGET}.")
budget_tab.round(4)

In [ ]:
snapshot_names = list(comparison_runs)
for page, start in enumerate(range(0, len(snapshot_names), 3), 1):
    names = snapshot_names[start:start + 3]
    page_runs = Runs({name: comparison_runs[name] for name in names})
    fig = plot_snapshots(page_runs, ring.p, ks=(0, 50, 150, N_R),
                         centers=ring.centers, extent=ring.extent)
    save(fig, f"04_sinkhorn_snapshots_{page}")
    display(fig); plt.close(fig)